# 🌉 SHMS AI Anomaly Detection — Finger Bridge
## Phase 3A: LSTM Autoencoder Training

**Jurnal target**: JCTA (Journal of Computing Theories and Applications) — SINTA 2

**Pipeline**:
```
GDrive (data .npy)  →  Colab GPU  →  Model .pt  →  GDrive (simpan balik)
```

---
### Cara pakai notebook ini
1. Pastikan data processed sudah ada di GDrive (`02_data/processed/*.npy`)
2. Jalankan **semua cell dari atas ke bawah** (Runtime → Run all)
3. Setelah selesai, model tersimpan otomatis ke GDrive

> **GPU**: Pastikan aktif → Runtime → Change runtime type → T4 GPU


---
## 0. Cek GPU & Environment

In [ ]:
# Cek GPU tersedia
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU tersedia:')
    # Ambil baris nama GPU saja
    for line in result.stdout.split('\n'):
        if 'Tesla' in line or 'T4' in line or 'A100' in line or 'V100' in line:
            print(' ', line.strip())
else:
    print('⚠️  GPU tidak terdeteksi — training akan pakai CPU (lebih lambat)')
    print('   Aktifkan: Runtime → Change runtime type → T4 GPU')


In [ ]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice yang digunakan: {device}')


---
## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive berhasil di-mount')


In [ ]:
from pathlib import Path

# ══════════════════════════════════════════════════════
# SESUAIKAN PATH INI dengan lokasi folder proyekmu di GDrive
# ══════════════════════════════════════════════════════
GDRIVE_PROJECT = Path('/content/drive/MyDrive/shms-ai-anomaly-detection-fingerbridge')

# Path folder
CODE_DIR      = GDRIVE_PROJECT / '03_code'
DATA_PROC_DIR = GDRIVE_PROJECT / '02_data' / 'processed'
MODEL_DIR     = GDRIVE_PROJECT / '04_models'
RESULTS_DIR   = GDRIVE_PROJECT / '05_results'

# Buat folder jika belum ada
for d in [MODEL_DIR, RESULTS_DIR, RESULTS_DIR/'figures']:
    d.mkdir(parents=True, exist_ok=True)

# Verifikasi
print('📁 Struktur folder:')
for label, path in [
    ('Code    ', CODE_DIR),
    ('Data    ', DATA_PROC_DIR),
    ('Models  ', MODEL_DIR),
    ('Results ', RESULTS_DIR),
]:
    exists = '✅' if path.exists() else '❌ TIDAK ADA'
    print(f'  {label}: {path}  {exists}')

# Cek file .npy tersedia
npy_files = sorted(DATA_PROC_DIR.glob('*_X.npy')) if DATA_PROC_DIR.exists() else []
print(f'\n📊 File processed tersedia: {len(npy_files)} hari')
if npy_files:
    print(f'   Pertama : {npy_files[0].name}')
    print(f'   Terakhir: {npy_files[-1].name}')


---
## 2. Install Dependencies & Import

In [ ]:
# Semua library sudah tersedia di Colab kecuali torch-geometric
# (torch-geometric hanya dibutuhkan untuk Phase 3C - GNN)
import subprocess
print('Mengecek library...')
libs = ['torch', 'numpy', 'pandas', 'sklearn', 'matplotlib', 'seaborn']
for lib in libs:
    try:
        mod = __import__(lib if lib != 'sklearn' else 'sklearn')
        ver = getattr(mod, '__version__', '?')
        print(f'  ✅ {lib:15s} {ver}')
    except ImportError:
        print(f'  ❌ {lib} tidak tersedia')


In [ ]:
import sys, json, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

# Tambahkan path code ke sys.path agar bisa import config
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print('✅ Semua library berhasil di-import')


---
## 3. Konfigurasi
> Edit cell ini jika ingin mengubah hyperparameter

In [ ]:
# ══════════════════════════════════════════════════════
# KONFIGURASI — edit sesuai kebutuhan
# ══════════════════════════════════════════════════════

# Channel sensor — 17 channel utama (accelerometer + cable)
# Sesuai dengan shms_config.py
MAIN_CHANNELS = [
    'FB_AC_PY1T_01_BX', 'FB_AC_PY1T_01_BY',
    'FB_AC_PY1D_01_BX', 'FB_AC_PY1D_01_BY',
    'FB_AC_S2M_01_BZ',
    'FB_AC_S1Q1_01_BX', 'FB_AC_S1Q1_01_BZ',
    'FB_AC_S3Q3_01_BY', 'FB_AC_S3Q3_01_BZ',
    'FB_CA_L17_EZ', 'FB_CA_R17_EZ',
    'FB_CA_L22_EZ', 'FB_CA_R22_EZ',
    'FB_CA_L46_EZ', 'FB_CA_R46_EZ',
    'FB_CA_L02_EZ', 'FB_CA_L55_EZ',
]
N_CHANNELS  = len(MAIN_CHANNELS)   # 17
WINDOW_SIZE = 1000                  # 10 detik @ 100Hz

# Hyperparameter model
HP = {
    # Arsitektur
    'n_channels'  : N_CHANNELS,
    'hidden_size' : 64,    # coba 128 jika hasil kurang baik
    'num_layers'  : 2,
    'dropout'     : 0.2,

    # Training
    'batch_size'  : 64,    # naikkan ke 128 jika RAM GPU cukup
    'lr'          : 1e-3,
    'n_epochs'    : 50,
    'patience'    : 7,     # early stopping
    'clip_grad'   : 1.0,

    # Threshold
    'threshold_pct': 95,   # persentil RE untuk batas anomali
}

print('✅ Konfigurasi:')
for k, v in HP.items():
    print(f'   {k:20s}: {v}')


---
## 4. Load Data Processed
> Data sudah dalam format .npy hasil dari batch processor

In [ ]:
def load_split(split: str, data_dir: Path,
               normal_only: bool = False,
               max_windows: int = None) -> tuple:
    """
    Load data .npy untuk satu split.
    Jika data belum ada, buat data simulasi untuk testing.
    """
    summary_path = data_dir / 'processing_summary.csv'

    # Mode simulasi jika data belum ada
    if not summary_path.exists() or not data_dir.exists():
        print(f'  ⚠️  [{split}] Data processed belum ada → mode simulasi')
        n = {'train': 5000, 'val': 1000, 'test': 1000}[split]
        X = np.random.randn(n, WINDOW_SIZE, N_CHANNELS).astype(np.float32)
        y = np.zeros(n, dtype=np.int8)
        if split != 'train':
            abn = np.random.choice(n, n//10, replace=False)
            X[abn] += np.random.randn(len(abn), WINDOW_SIZE, N_CHANNELS).astype(np.float32) * 3
            y[abn]  = 1
        print(f'  [{split}] Simulasi: {len(X):,} windows')
        return X, y

    summary = pd.read_csv(summary_path)
    days    = summary[
        (summary['split'] == split) & (summary['status'] == 'ok')
    ]['date'].tolist()

    X_list, y_list = [], []
    for d in sorted(days):
        xp = data_dir / f'{d}_X.npy'
        yp = data_dir / f'{d}_y.npy'
        if xp.exists():
            X_list.append(np.load(xp))
            y_list.append(np.load(yp))

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    y = np.concatenate(y_list, axis=0)

    if normal_only:
        mask = y == 0
        X, y = X[mask], y[mask]

    if max_windows and len(X) > max_windows:
        idx  = np.sort(np.random.choice(len(X), max_windows, replace=False))
        X, y = X[idx], y[idx]

    return X, y


In [ ]:
print('📂 Loading data...')
X_train, y_train = load_split('train', DATA_PROC_DIR, normal_only=True)
X_val,   y_val   = load_split('val',   DATA_PROC_DIR, normal_only=False)
X_test,  y_test  = load_split('test',  DATA_PROC_DIR, normal_only=False)

print(f'\n📊 Dataset summary:')
print(f'  Train  : {X_train.shape}  — hanya normal (untuk training)')
print(f'  Val    : {X_val.shape}  — {y_val.sum()} abnormal ({100*y_val.mean():.1f}%)')
print(f'  Test   : {X_test.shape}  — {y_test.sum()} abnormal ({100*y_test.mean():.1f}%)')
print(f'  Channels: {X_train.shape[2]} | Window: {X_train.shape[1]} sampel')

# Estimasi ukuran di memori
total_mb = (X_train.nbytes + X_val.nbytes + X_test.nbytes) / 1024**2
print(f'\n  Total RAM terpakai: {total_mb:.0f} MB')


---
## 5. Definisi Model LSTM Autoencoder

In [ ]:
class LSTMAutoencoder(nn.Module):
    """
    LSTM Autoencoder untuk anomaly detection time-series.

    Encoder: kompres (batch, 1000, 17) → hidden state
    Decoder: rekonstruksi hidden state → (batch, 1000, 17)
    Loss   : MSE(input, rekonstruksi)

    Anomali dideteksi jika reconstruction error > threshold.
    """
    def __init__(self, n_channels, hidden_size, num_layers, dropout):
        super().__init__()
        self.encoder = nn.LSTM(
            input_size  = n_channels,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )
        self.decoder = nn.LSTM(
            input_size  = hidden_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )
        self.output_layer = nn.Linear(hidden_size, n_channels)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        _, (hidden, cell)   = self.encoder(x)
        dec_input = hidden[-1].unsqueeze(1).repeat(1, seq_len, 1)
        dec_out, _          = self.decoder(dec_input, (hidden, cell))
        return self.output_layer(dec_out)

    def reconstruction_error(self, x, x_hat):
        """Per-window MSE. Shape output: (batch,)"""
        return ((x - x_hat) ** 2).mean(dim=(1, 2))


# Inisialisasi model
model = LSTMAutoencoder(
    n_channels  = HP['n_channels'],
    hidden_size = HP['hidden_size'],
    num_layers  = HP['num_layers'],
    dropout     = HP['dropout'],
).to(device)

# Hitung jumlah parameter
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model berhasil dibuat')
print(f'   Total parameter : {n_params:,}')
print(f'   Estimasi ukuran : {n_params*4/1024:.0f} KB')
print(f'\nArsitektur:')
print(model)


---
## 6. Training

> **Estimasi waktu**:
> - CPU : ~3–5 menit per epoch → total ~2–4 jam
> - T4 GPU : ~15–30 detik per epoch → total ~15–25 menit


In [ ]:
# DataLoader — hemat RAM, batch per batch
train_dl = DataLoader(
    TensorDataset(torch.FloatTensor(X_train)),
    batch_size=HP['batch_size'], shuffle=True, num_workers=2, pin_memory=True
)
val_dl = DataLoader(
    TensorDataset(torch.FloatTensor(X_val)),
    batch_size=HP['batch_size'], shuffle=False, num_workers=2
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=HP['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.5, verbose=True
)

print(f'Train batches : {len(train_dl)}')
print(f'Val batches   : {len(val_dl)}')
print(f'Batch size    : {HP["batch_size"]}')
print(f'Max epochs    : {HP["n_epochs"]}')
print(f'Early stopping: patience={HP["patience"]}')


In [ ]:
# ── TRAINING LOOP ──────────────────────────────────────
history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
patience_cnt  = 0
best_state    = None

print(f'{'='*65}')
print(f'  MULAI TRAINING — device: {device}')
print(f'{'='*65}')

for epoch in range(1, HP['n_epochs'] + 1):
    t0 = time.time()

    # Train
    model.train()
    train_loss = 0.0
    for (bx,) in train_dl:
        bx = bx.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), bx)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), HP['clip_grad'])
        optimizer.step()
        train_loss += loss.item() * len(bx)
    train_loss /= len(X_train)

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for (bx,) in val_dl:
            bx = bx.to(device)
            val_loss += criterion(model(bx), bx).item() * len(bx)
    val_loss /= len(X_val)

    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    elapsed = time.time() - t0

    mark = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = {k: v.cpu().clone() for k,v in model.state_dict().items()}
        patience_cnt  = 0
        mark = ' ← best ✓'
    else:
        patience_cnt += 1

    print(f'Epoch {epoch:3d}/{HP["n_epochs"]} | '
          f'train={train_loss:.6f} | val={val_loss:.6f} | '
          f'{elapsed:.1f}s{mark}')

    if patience_cnt >= HP['patience']:
        print(f'\nEarly stopping — epoch {epoch}')
        break

# Restore best
model.load_state_dict(best_state)
print(f'\n✅ Training selesai | best val_loss = {best_val_loss:.6f}')


---
## 7. Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(history['train_loss'], label='Train loss', color='#378ADD', lw=1.8)
ax.plot(history['val_loss'],   label='Val loss',   color='#D85A30', lw=1.8)
best_ep = history['val_loss'].index(min(history['val_loss']))
ax.axvline(best_ep, color='#BA7517', lw=1.2, linestyle='--',
           label=f'Best epoch={best_ep+1}')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('LSTM Autoencoder — Training & Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plot_path = RESULTS_DIR / 'figures' / 'lstm_loss_curve.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Disimpan: {plot_path}')


---
## 8. Kalibrasi Threshold Anomali

Threshold dihitung dari **persentil ke-N** dari distribusi
reconstruction error (RE) pada data normal di validation set.

- RE > threshold → **ANOMALI**
- RE ≤ threshold → **NORMAL**


In [ ]:
model.eval()
all_re_val = []
val_dl_full = DataLoader(
    TensorDataset(torch.FloatTensor(X_val)),
    batch_size=HP['batch_size'], shuffle=False
)
with torch.no_grad():
    for (bx,) in val_dl_full:
        bx = bx.to(device)
        x_hat = model(bx)
        re    = model.reconstruction_error(bx, x_hat)
        all_re_val.extend(re.cpu().numpy())

all_re_val  = np.array(all_re_val)
re_normal   = all_re_val[y_val == 0]
threshold   = np.percentile(re_normal, HP['threshold_pct'])

print(f'Distribusi RE (data normal):')
print(f'  Mean  : {re_normal.mean():.6f}')
print(f'  Std   : {re_normal.std():.6f}')
print(f'  P75   : {np.percentile(re_normal, 75):.6f}')
print(f'  P90   : {np.percentile(re_normal, 90):.6f}')
print(f'  P95   : {np.percentile(re_normal, 95):.6f}  ← threshold default')
print(f'  P99   : {np.percentile(re_normal, 99):.6f}')
print(f'\n✅ Threshold dipilih (P{HP["threshold_pct"]}): {threshold:.6f}')


In [ ]:
# Plot distribusi RE
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
re_abnorm = all_re_val[y_val == 1] if y_val.sum() > 0 else np.array([])
axes[0].hist(re_normal, bins=60, alpha=0.75, color='#378ADD', label='Normal')
if len(re_abnorm):
    axes[0].hist(re_abnorm, bins=40, alpha=0.75, color='#E24B4A', label='Abnormal')
axes[0].axvline(threshold, color='#BA7517', lw=2, linestyle='--',
                label=f'Threshold P{HP["threshold_pct"]}={threshold:.5f}')
axes[0].set_xlabel('Reconstruction Error (MSE)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribusi RE — Validation Set')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Persentil curve
pcts  = np.arange(50, 100, 0.5)
vals  = [np.percentile(re_normal, p) for p in pcts]
axes[1].plot(pcts, vals, color='#378ADD', lw=1.8)
axes[1].axvline(HP['threshold_pct'], color='#BA7517', lw=1.5,
                linestyle='--', label=f'P{HP["threshold_pct"]}')
axes[1].axhline(threshold, color='#BA7517', lw=1, linestyle=':')
axes[1].set_xlabel('Persentil')
axes[1].set_ylabel('RE value')
axes[1].set_title('Kurva Persentil RE Normal')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('LSTM Autoencoder — Threshold Calibration', fontsize=12)
plt.tight_layout()
plot_path = RESULTS_DIR / 'figures' / 'lstm_re_distribution.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Disimpan: {plot_path}')


---
## 9. Evaluasi pada Data Test

In [ ]:
# Hitung RE pada data test
test_dl = DataLoader(
    TensorDataset(torch.FloatTensor(X_test)),
    batch_size=HP['batch_size'], shuffle=False
)
model.eval()
all_re_test = []
with torch.no_grad():
    for (bx,) in test_dl:
        bx = bx.to(device)
        x_hat = model(bx)
        re    = model.reconstruction_error(bx, x_hat)
        all_re_test.extend(re.cpu().numpy())

all_re_test = np.array(all_re_test)
y_pred      = (all_re_test > threshold).astype(int)

# Normalisasi ke [0,1] sebagai anomaly score
re_min, re_max = all_re_test.min(), all_re_test.max()
scores_test    = (all_re_test - re_min) / (re_max - re_min + 1e-9)

# Metrics
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
auc       = roc_auc_score(y_test, scores_test) if y_test.sum() > 0 else 0.0
cm        = confusion_matrix(y_test, y_pred)

print('='*50)
print('  HASIL EVALUASI — DATA TEST')
print('='*50)
print(f'  Precision  : {precision:.4f}')
print(f'  Recall     : {recall:.4f}')
print(f'  F1-score   : {f1:.4f}')
print(f'  AUC-ROC    : {auc:.4f}')
print(f'\n  Confusion Matrix:')
print(f'    TN={cm[0,0]:6,}  FP={cm[0,1]:6,}')
print(f'    FN={cm[1,0]:6,}  TP={cm[1,1]:6,}')
print('='*50)


In [ ]:
# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Confusion matrix
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['Normal','Abnormal'])
axes[0].set_yticklabels(['Normal','Abnormal'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                     fontsize=14, fontweight='bold',
                     color='white' if cm[i,j] > cm.max()//2 else 'black')

# ROC curve
if y_test.sum() > 0:
    fpr, tpr, _ = roc_curve(y_test, scores_test)
    axes[1].plot(fpr, tpr, color='#378ADD', lw=2, label=f'AUC = {auc:.4f}')
    axes[1].plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.suptitle('LSTM Autoencoder — Evaluasi Data Test', fontsize=12)
plt.tight_layout()
plot_path = RESULTS_DIR / 'figures' / 'lstm_evaluation.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Disimpan: {plot_path}')


---
## 10. Anomaly Detection Timeline

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

idx = np.arange(len(all_re_test))

axes[0].plot(idx, all_re_test, color='#888780', lw=0.6, alpha=0.8)
axes[0].axhline(threshold, color='#BA7517', lw=1.5, linestyle='--',
                label=f'Threshold={threshold:.5f}')
if y_test.sum():
    abn_idx = np.where(y_test == 1)[0]
    axes[0].scatter(abn_idx, all_re_test[abn_idx], color='#E24B4A',
                    s=6, zorder=5, alpha=0.8, label='Actual abnormal')
fp_idx = np.where((y_pred == 1) & (y_test == 0))[0]
if len(fp_idx):
    axes[0].scatter(fp_idx, all_re_test[fp_idx], color='#FAC775',
                    s=4, zorder=4, alpha=0.6, label='False positive')
axes[0].set_ylabel('Reconstruction Error')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.2)

axes[1].fill_between(idx, scores_test, alpha=0.6,
                     color='#378ADD', label='Anomaly score')
axes[1].axhline(0.5, color='#BA7517', lw=1, linestyle='--', alpha=0.7)
axes[1].set_ylabel('Anomaly Score (0–1)')
axes[1].set_xlabel('Window index (urutan waktu)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.2)

fig.suptitle('LSTM Autoencoder — Anomaly Detection Timeline (Test Set)',
             fontsize=12)
plt.tight_layout()
plot_path = RESULTS_DIR / 'figures' / 'lstm_anomaly_timeline.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()


---
## 11. Simpan Model ke GDrive

In [ ]:
import json

# Simpan model
model_path = MODEL_DIR / 'lstm_autoencoder_best.pt'
torch.save({
    'model_state'     : {k: v.cpu() for k,v in model.state_dict().items()},
    'hp'              : HP,
    'best_val_loss'   : best_val_loss,
    'n_epochs_trained': epoch,
    'metrics'         : {'precision': precision, 'recall': recall,
                         'f1': f1, 'auc': auc},
}, model_path)

# Simpan threshold
thresh_path = MODEL_DIR / 'lstm_threshold.json'
with open(thresh_path, 'w') as f:
    json.dump({
        'threshold'      : float(threshold),
        'threshold_pct'  : HP['threshold_pct'],
        're_mean_normal' : float(re_normal.mean()),
        're_std_normal'  : float(re_normal.std()),
    }, f, indent=2)

# Simpan metrics ke CSV
metrics_row = {
    'model': 'LSTM_Autoencoder', 'threshold': threshold,
    'precision': precision, 'recall': recall,
    'f1': f1, 'auc': auc,
    'tn': cm[0,0], 'fp': cm[0,1], 'fn': cm[1,0], 'tp': cm[1,1],
    'n_test_windows': len(y_test), 'n_test_abnormal': int(y_test.sum()),
    'hidden_size': HP['hidden_size'], 'num_layers': HP['num_layers'],
    'epochs_trained': epoch, 'best_val_loss': best_val_loss,
}
pd.DataFrame([metrics_row]).to_csv(
    RESULTS_DIR / 'lstm_metrics.csv', index=False
)

# Update experiment log
log_path = GDRIVE_PROJECT / '05_results' / 'experiment_log.csv'
log_row  = pd.DataFrame([{
    'run_id'     : pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'),
    'timestamp'  : pd.Timestamp.now().isoformat(),
    'model'      : 'LSTM_Autoencoder',
    'window_size': 1000,
    'threshold'  : round(threshold, 6),
    'precision'  : round(precision, 4),
    'recall'     : round(recall,    4),
    'f1'         : round(f1,        4),
    'auc'        : round(auc,       4),
    'notes'      : f'hidden={HP["hidden_size"]}, layers={HP["num_layers"]}',
}])
if log_path.exists():
    log_row.to_csv(log_path, mode='a', header=False, index=False)
else:
    log_row.to_csv(log_path, index=False)

print('✅ Semua file tersimpan ke GDrive:')
for p in [model_path, thresh_path,
          RESULTS_DIR/'lstm_metrics.csv', log_path]:
    size = p.stat().st_size/1024 if p.exists() else 0
    print(f'   {p.name:40s} {size:6.1f} KB')


---
## 12. Ringkasan Hasil

In [ ]:
print('='*55)
print('  RINGKASAN — LSTM AUTOENCODER')
print('='*55)
print(f'  Precision  : {precision:.4f}')
print(f'  Recall     : {recall:.4f}')
print(f'  F1-score   : {f1:.4f}  ← metrik utama')
print(f'  AUC-ROC    : {auc:.4f}')
print(f'  Threshold  : {threshold:.6f} (P{HP["threshold_pct"]})')
print(f'  Epochs     : {epoch}')
print(f'  Val Loss   : {best_val_loss:.6f}')
print('='*55)
print(f'\n  File tersimpan di GDrive:')
print(f'  04_models/lstm_autoencoder_best.pt')
print(f'  04_models/lstm_threshold.json')
print(f'  05_results/lstm_metrics.csv')
print(f'  05_results/figures/lstm_*.png (4 plot)')
print(f'\n  Next: Phase 3B — Isolation Forest')
print(f'        notebook: nb_phase3b_iforest.ipynb')
